In [1]:
import pandas as pd  
import numpy as np  
import matplotlib.pyplot as plt  
import seaborn as sns  
from matplotlib_venn import venn2  
from scipy import stats  
import os  
from matplotlib.lines import Line2D  
from matplotlib import rcParams  

# Create output directory for plots  
output_dir = '../comparison/hSyn1_GW_4M_vs_ineuron_crispr_comparison_plots'  
if not os.path.exists(output_dir):  
    os.makedirs(output_dir)  

In [2]:
rcParams['font.family'] = 'sans-serif'  
rcParams['font.sans-serif'] = ['Arial']  
rcParams['font.size'] = 10
rcParams['axes.linewidth'] = 0.5  
rcParams['xtick.major.width'] = 0.8  
rcParams['ytick.major.width'] = 0.8  
rcParams['xtick.major.size'] = 2 
rcParams['ytick.major.size'] = 2  
rcParams['pdf.fonttype'] = 42  # Ensures text is editable in AI  
rcParams['ps.fonttype'] = 42  
rcParams['svg.fonttype'] = 'none'  
rcParams['figure.dpi'] = 300  
rcParams['ytick.labelsize'] = 8
rcParams['xtick.labelsize'] = 8

In [3]:
screen1_data = pd.read_csv('../new_iNC/Syn1-GW-all_4M/Syn1-GW-all_4M_all_genes.csv')
from gseapy import Biomart
bm = Biomart()
# note the dataset and attribute names are different
m2h = bm.query(dataset='mmusculus_gene_ensembl',
               attributes=['ensembl_gene_id','external_gene_name',
                           'hsapiens_homolog_ensembl_gene',
                           'hsapiens_homolog_associated_gene_name'])
mapping = dict(zip(m2h['external_gene_name'], m2h['hsapiens_homolog_associated_gene_name']))
screen1_data['gene'] = [mapping[i] if i in mapping.keys() else i for i in screen1_data['gene'].tolist()]

# screen2_data = pd.read_csv('../analysis/iNeuron_plusAO.csv')
screen2_data = pd.read_csv('../related_files/iNeuron_noAO.csv')
df_mapping2 = pd.read_table('CRISPRi_v2_name_updated.txt', header=None)
mapping2= dict(zip(df_mapping2[2], df_mapping2[5]))
screen2_data['gene'] = [mapping2[i] if i in mapping2.keys() else i for i in screen2_data['gene'].tolist()]

In [ ]:
merged_data = pd.merge(  
    screen1_data,   
    screen2_data,   
    on='gene',   
    how='inner',   
    suffixes=('_screen1', '_screen2')  
)  


In [ ]:
merged_data = merged_data.fillna({  
    'epsilon_screen1': 0,   
    'epsilon_screen2': 0,  
    'pvalue_screen1': 1,  
    'pvalue_screen2': 1,   
    'product_screen1': 0,  
    'product_screen2': 0  
})  

print(f"Merged data: {len(merged_data)} genes")  

In [ ]:
to_label = ['FAM50A', 'GPX4']
df_to_label = merged_data[merged_data['gene'].isin(to_label)]

In [ ]:
screen1 = 'iNeuron_-AO'
screen2 = 'hSyn1_GW'
plt.figure(figsize=[2,2])

plt.scatter(merged_data['product_screen2'], merged_data['product_screen1'],c='grey',
            s=5, alpha = 1, linewidth=0.2,edgecolors = 'white')
plt.plot([0, 0],[-4.5, 1.2] , 'k-', linewidth=0.5)
plt.plot([-14, 7], [0, 0], 'k-', linewidth=0.5)


plt.scatter(df_to_label['product_screen2'], df_to_label['product_screen1'],c='red',
            s=20, linewidths=0.2,edgecolors = 'k', zorder=2.5)
for gene, x, y in zip(df_to_label['gene'], df_to_label['product_screen2'], df_to_label['product_screen1']):
    plt.annotate(gene, (x,y))
    
plt.ylim([-3.5, 1.2])
plt.xlim([-14, 6])
plt.xlabel(screen1)
plt.ylabel(screen2)
plt.savefig(f'{output_dir}/{screen1}_vs_{screen2}.pdf', bbox_inches = 'tight')